# Task 4 runner for Colab (browser) or Kaggle

This single notebook runs all of Task 4 on a cloud GPU:
1. unpacks the code from `pa1_task4_bundle.zip`;
2. (Colab only) mounts Google Drive, so checkpoints, figures and executed notebooks are saved to `MyDrive/pa1_task4/`;
3. runs `1_vanilla_baseline → 3_gcsc → 4_proser → 2_posthoc_scores → 5_evaluation` in that order (the manual's order: unknowns only after every model is trained), printing training progress as it goes;
4. keeps each executed notebook, with all its outputs and plots.

The datasets download automatically inside the notebooks (CIFAR-10 in 1, CIFAR-100 in 2).

**Resuming after a disconnect:** trained checkpoints are kept (Drive on Colab, the output folder on Kaggle). Set `START_FROM` below to the first notebook that did **not** finish, then run again. Later notebooks load the saved models instead of retraining.

In [ ]:
# ---- settings ----
START_FROM = '1_vanilla_baseline'      # e.g. '3_gcsc' to resume after 1 finished
ORDER = ['1_vanilla_baseline', '3_gcsc', '4_proser', '2_posthoc_scores', '5_evaluation']

import os, sys, glob, zipfile, shutil, subprocess, time
ON_KAGGLE = os.path.isdir('/kaggle/input') or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
# Kaggle images also ship the google.colab package, so test for a real Colab machine instead
ON_COLAB = (not ON_KAGGLE) and (os.path.exists('/var/colab/hostname') or 'COLAB_RELEASE_TAG' in os.environ)
print('platform:', 'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'other')

import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU: {torch.cuda.get_device_name(0)} (compute {cap[0]}.{cap[1]}) | precision: {"bf16" if cap[0] >= 8 else "fp16 + loss scaling"}')
else:
    raise SystemExit('No GPU. Colab: Runtime -> Change runtime type -> GPU. Kaggle: Settings -> Accelerator -> GPU.')

In [ ]:
# ---- find and unpack the code bundle ----
WORK = '/kaggle/working/pa1' if ON_KAGGLE else '/content/pa1'
cands = (glob.glob('/kaggle/input/**/pa1_task4_bundle.zip', recursive=True) + glob.glob('/content/pa1_task4_bundle.zip')
         + glob.glob('/content/drive/MyDrive/pa1_task4_bundle.zip'))
if not cands and ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    cands = glob.glob('/content/drive/MyDrive/pa1_task4_bundle.zip')
if cands:
    with zipfile.ZipFile(cands[0]) as z:
        z.extractall(WORK)
    print('code unpacked from', cands[0], '->', WORK)
else:
    # Kaggle unzips uploaded datasets: look for the extracted folders instead
    src = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/task4_open_set/src', recursive=True)]
    assert src, ('Code not found. Colab: upload pa1_task4_bundle.zip with the Files panel (left sidebar) or put it in MyDrive. '
                 'Kaggle: add the dataset you made from pa1_task4_bundle.zip (right panel -> Add Input).')
    shutil.copytree(src[0], WORK, dirs_exist_ok=True)       # /kaggle/input is read-only, so copy
    print('code copied from', src[0], '->', WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# ---- where results are kept ----
T4 = f'{WORK}/task4_open_set'
if ON_COLAB:
    from google.colab import drive
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    KEEP = '/content/drive/MyDrive/pa1_task4'
    for sub in ['checkpoints', 'cache', 'results', 'splits', 'executed_notebooks']:
        os.makedirs(f'{KEEP}/{sub}', exist_ok=True)
    for sub in ['checkpoints', 'cache', 'results', 'splits']:
        link = f'{T4}/{sub}'
        if os.path.isdir(link) and not os.path.islink(link):
            subprocess.run(f'cp -rn "{link}/." "{KEEP}/{sub}/"', shell=True)
            shutil.rmtree(link)
        if not os.path.exists(link):
            os.symlink(f'{KEEP}/{sub}', link)
    print('outputs are written to Google Drive:', KEEP)
else:
    KEEP = T4
    print('outputs are written to', KEEP, '(Kaggle keeps /kaggle/working as the notebook output)')

In [ ]:
# ---- download the datasets up front, with visible progress ----
# (the notebooks' own download cells then find the verified files and skip the download;
#  CIFAR-100 is only fetched here, it is first *used* in 2_posthoc_scores after all models are trained)
from torchvision import datasets

def stage_from_inputs(needed, dest):
    """Copy official CIFAR 'python version' files found anywhere under /kaggle/input into `dest`
    (the layout torchvision expects). torchvision then verifies every file's md5 below; if a copy is not
    the official data, torchvision downloads the official archive instead."""
    if not os.path.isdir('/kaggle/input') or os.path.isdir(dest):
        return
    for d, _, files in os.walk('/kaggle/input'):
        if set(needed) <= set(files):
            os.makedirs(dest, exist_ok=True)
            for f in needed:
                shutil.copy(os.path.join(d, f), dest)
            print(f'using Kaggle input {d} -> {dest}', flush=True)
            return

stage_from_inputs(['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch', 'batches.meta'],
                  f'{WORK}/data/cifar10/cifar-10-batches-py')
stage_from_inputs(['train', 'test', 'meta'], f'{WORK}/data/cifar100/cifar-100-python')
for name, fn, root, kw in [('CIFAR-10 train', datasets.CIFAR10, 'cifar10', dict(train=True)),
                           ('CIFAR-10 test', datasets.CIFAR10, 'cifar10', dict(train=False)),
                           ('CIFAR-100 test', datasets.CIFAR100, 'cifar100', dict(train=False))]:
    t0 = time.time()
    ds = fn(f'{WORK}/data/{root}', download=True, **kw)
    print(f'{name}: {len(ds)} images ready ({time.time() - t0:.0f}s)', flush=True)

In [ ]:
# ---- run the Task 4 notebooks in order ----
LOG = f'{WORK}/train_progress.log'
env = dict(os.environ, PA1_TRAIN_LOG=LOG, PYTHONWARNINGS='ignore')
todo = ORDER[ORDER.index(START_FROM):]
for name in todo:
    nb = f'{T4}/notebooks/{name}.ipynb'
    print(f'\n=== {time.strftime("%H:%M:%S")} running {name} ===', flush=True)
    open(LOG, 'a').close(); pos = os.path.getsize(LOG)
    p = subprocess.Popen([sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute', '--inplace',
                          '--ExecutePreprocessor.timeout=-1', '--ExecutePreprocessor.kernel_name=python3', nb],
                         cwd=f'{T4}/notebooks', env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    t_start = last_msg = time.time()
    while p.poll() is None:
        time.sleep(15)
        with open(LOG) as f:
            f.seek(pos); new = f.read(); pos = f.tell()
        if new:
            print(new, end='', flush=True); last_msg = time.time()
        elif time.time() - last_msg > 120:                    # heartbeat so silence never looks like a hang
            print(f'  ... still running {name} ({(time.time() - t_start) / 60:.0f} min elapsed)', flush=True)
            last_msg = time.time()
    tail = p.stdout.read()
    if ON_COLAB:
        shutil.copy(nb, f'{KEEP}/executed_notebooks/{name}.ipynb')
    if p.returncode != 0:
        print(tail[-4000:])
        raise SystemExit(f'{name} failed (see the error above). Fix, then set START_FROM = "{name}" and re-run this cell.')
    print(f'=== {time.strftime("%H:%M:%S")} {name} finished ===', flush=True)
print('\nAll Task 4 notebooks finished.')

In [ ]:
# ---- tidy up (Kaggle): drop the downloaded datasets so the output only holds results ----
if ON_KAGGLE:
    shutil.rmtree(f'{WORK}/data', ignore_errors=True)
for sub in ['checkpoints', 'cache', 'results']:
    d = f'{T4}/{sub}'
    if os.path.isdir(d):
        print(sub, len(os.listdir(d)), 'files')
print('executed notebooks:', f'{KEEP}/executed_notebooks' if ON_COLAB else f'{T4}/notebooks')